# Part 1: Build and analyze a deep research agent

This notebook runs and plots your Part 1 work. The [Part 1 handout](../docs/part1_handout.md) is the specification.

Use the **CS 378 Assignment 4** kernel ([setup](../README.md#setup)).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())
plt.rcParams["figure.dpi"] = 100

BACKEND = "bedrock" if os.environ.get("AGENTKV_BEDROCK_MODEL") else "fake"
print(f"Working in {Path.cwd()}")
hint = "" if BACKEND == "bedrock" else "  (set AGENTKV_BEDROCK_MODEL to use Bedrock)"
print(f"LLM backend: {BACKEND}{hint}")

## 1. Corpus and tools

The agent searches an offline corpus of 48 Wikipedia articles on six systems topics, each saved at two revisions (`v1` and `v2`), through two tools:

- `search(query, k=5)` ranks articles by BM25 and returns up to `k` hits with their `doc_id`, title, score, and a snippet.
- `fetch(doc_id)` returns the article text and revision metadata.

Inside the agent, both tools add seeded delays, and in the failure run, transient failures.

In [ ]:
from research_app.corpus import Corpus
from research_app.questions import load_questions

corpus = Corpus("v2")
print({topic: len(ids) for topic, ids in corpus.topics().items()})

for hit in corpus.search("how does raft elect a leader", k=3):
    print(f"{hit.doc_id:40} score {hit.score:.2f}  {hit.title}")

In [ ]:
doc = corpus.fetch(corpus.search("how does raft elect a leader", k=1)[0].doc_id)
print(doc.title, f"({len(doc.text)} characters, snapshot sha256 {doc.sha256[:12]})")
print(doc.text[:600], "...")

The first five of the 16 public questions. Try a few queries of your own with `corpus.search()`.

In [ ]:
questions = load_questions(public=True)
for q in questions[:5]:
    print(f"{q['id']:18} {q['question']}")
print(f"... {len(questions)} questions")

## 2. Concurrency cap (1.1)

All 16 reports share one cap C on concurrent LLM calls. Implement `acquire`, `release`, and `set_limit` on `AdjustableCap` in `research_app/concurrency.py`, then run the tests.

In [ ]:
check("tests/test_part1_concurrency.py")

Six calls start under a limit of 3, and the limit drops to 1 while three are running. Correct behavior: those three finish, the other three run one at a time, the peak is 3, and no call starts over the limit. Yellow is waiting for a slot; blue is running.

In [ ]:
from research_app.concurrency import AdjustableCap
from research_app.visualize import run_cap_demo, plot_cap_demo

cap = AdjustableCap(3)
demo = await run_cap_demo(cap)
plot_cap_demo(demo)
plt.show()
print(f"Peak running calls: {demo.peak} (expected 3)")
print(f"Calls incorrectly started at or above the limit: {demo.over_limit} (expected 0)")

## 3. Research agent (1.2)

Implement `research_app/agent.py`, starting with `build_graph`. The handout has the LangGraph concepts and the `op_id` and `order` table; the TODO comment at each node specifies it. The cell below runs one report and stops with `NotImplementedError` at the first node you have not written.

![Research graph: plan, parallel research branches, coverage check, optional follow-up, synthesize, validate, and optional repair.](../docs/figures/research_workflow.svg)

In [ ]:
from IPython.display import Markdown, display
from research_app import ResearchAgent
from research_app.llm import make_llm
from research_app.search import LocalSearch

agent = ResearchAgent(llm=make_llm(BACKEND), search=LocalSearch(), cap=2)
result = await agent.research(questions[0]["question"])
print(f"Citation checks passed: {result.valid}; LLM calls: {result.recording.counts()['llm_calls']}")
print("Errors:", result.recording.errors)

In [ ]:
display(Markdown(result.markdown()))

In [ ]:
check("tests/test_part1_agent.py", "tests/test_part1_citations.py", "tests/test_part1_llm.py",
      "tests/test_part1_trace_export.py")

## 4. Run the agent (1.4)

Three runs of the 16 public questions, one report starting every 2 seconds: C = 2, C = 8, and C = 8 with 20% of tool attempts failing. They use Bedrock if `AGENTKV_BEDROCK_MODEL` is set and `FakeLLM` otherwise; record which. A `FakeLLM` run takes one to two minutes. Every run must print `16/16 valid`.

In [ ]:
!{sys.executable} -m research_app.run --backend {BACKEND} --cap 2 --replay --out runs/c2
!{sys.executable} -m research_app.run --backend {BACKEND} --cap 8 --replay --out runs/c8
FAULTS = "--failure-rate 0.2"
!{sys.executable} -m research_app.run --backend {BACKEND} --cap 8 {FAULTS} --replay --out runs/c8-faults

## 5. Workload analysis (1.3)

Implement the four functions in `research_app/analysis.py` and run the tests. The plots below use them.

In [ ]:
check("tests/test_part1_analysis.py")

### 5a. Critical path

The DAG highlights one C = 2 report's critical path; each box shows the operation's service time and, for LLM calls, its slot wait. The bar chart breaks down every report's path in the three runs. Use both for Q1.

In [ ]:
from research_app.visualize import plot_critical_paths, plot_research_dag

fig, steps = plot_research_dag("runs/c2", report_id="R04")   # choose any report
plt.show()
fig, totals = plot_critical_paths(["runs/c2", "runs/c8", "runs/c8-faults"])
plt.show()
for row in totals:
    print(f"{row['run']}: mean per report: slot wait {row['wait']:.1f} s, LLM {row['llm']:.1f} s, "
          f"tools {row['tool']:.1f} s")

### 5b. Prefix reuse

Prefix reuse by stage, and how long reused blocks sat unused before their reuse. Use these for Q2.

In [ ]:
from research_app.analysis import load_run, shared_prefix_share
from research_app.visualize import plot_prefix_reuse, plot_reuse_gaps

for run_dir in ("runs/c2", "runs/c8"):
    calls = load_run(run_dir).llm_calls()
    print(f"{run_dir}: {len(calls)} LLM calls; shared prefix share {shared_prefix_share(calls):.1%}")
    plot_prefix_reuse(calls, shared_prefix_share)
    plt.show()
plot_reuse_gaps(["runs/c2", "runs/c8"])
plt.show()

### 5c. Concurrency and memory

Latency, slot waits, and each report's timeline at C = 2 and C = 8 (yellow: waiting for a slot; blue: running; gray: tools), then the KV footprint each run would need on one GPU.

In [ ]:
from research_app.visualize import plot_kv_footprint, plot_run_comparison, plot_run_timelines

fig, measurements = plot_run_comparison(["runs/c2", "runs/c8"])
plt.show()
plot_run_timelines(["runs/c2", "runs/c8"])
plt.show()
fig, peaks = plot_kv_footprint(["runs/c2", "runs/c8"])
plt.show()
for row, peak in zip(measurements, peaks):
    print(f"C = {row['cap']}: {row['valid']}/{row['reports']} valid; "
          f"peak {row['llm_peak']} concurrent LLM calls; "
          f"mean slot wait {row['mean_slot_wait']:.2f} s; peak KV {peak['private_gb']:.1f} GB "
          f"({peak['shared_gb']:.1f} GB sharing prefixes)")

The hosted model hides memory pressure, so replay your C = 8 run in the simulator on one A100 at several caps and KV capacities. The replay keeps your run's dependencies, prompts, outputs, and tool delays, and uses `policies/fifo.py`. Use the sweep for Q3.

In [ ]:
caps = (1, 2, 4, 8, 16)
kv_sizes = (12_288, 16_384, 49_152, 393_216)       # tokens; 128 KiB each
latency = {}
for kv in kv_sizes:
    for cap in caps:
        rec = viz.record("runs/c8/replay.json", "policies/fifo.py", params={"cap": cap}, kv_tokens=kv)
        latency[kv, cap] = sum(rec.flows.values()) / len(rec.flows)

fig, ax = plt.subplots(figsize=(7, 3.6))
for kv in kv_sizes:
    label = f"{kv:,} tokens ({kv * 131072 / 1e9:.1f} GB)"
    ax.plot(caps, [latency[kv, c] for c in caps], marker="o", label=label)
ax.set(xscale="log", xticks=caps, xticklabels=caps, xlabel="Concurrency cap",
       ylabel="Mean report latency (s)")
ax.legend(title="KV-cache capacity", frameon=False)
plt.show()

### 5d. Failures

Compare `runs/c8-faults` with `runs/c8` for Q4.

In [ ]:
from research_app.analysis import critical_path

for run_dir in ("runs/c8", "runs/c8-faults"):
    run = load_run(run_dir)
    ops = run.ops.values()
    latencies = sorted(max(o.end for o in run.report_ops(r)) - run.arrivals[r] for r in run.arrivals)
    on_path = {s.op_id for r in run.arrivals for s in critical_path(run, r)}
    retried = [o for o in ops if o.attempts > 1]
    print(f"{run_dir}: median latency {latencies[len(latencies) // 2]:.1f} s, max {latencies[-1]:.1f} s; "
          f"{len(retried)} retried operations "
          f"({sum(o.op_id in on_path for o in retried)} on a critical path), "
          f"{sum('/followup/' in o.op_id for o in ops if o.kind == 'llm')} follow-ups, "
          f"{sum(o.op_id.endswith('/repair') for o in ops)} repairs")

## Questions

Answer Q1–Q4 from the [handout](../docs/part1_handout.md#14-runs-and-questions) in `assignment4_report.pdf`.

## 6. Optional: research on the live web

This section is not graded. It runs the same agent with `BedrockLLM()` and a web-search adapter from `research_app/search.py` instead of the local corpus. Set your Bedrock variables and one provider's key before starting Jupyter, then set `RUN_WEB_RESEARCH = True`.

| Provider | Adapter and environment variable | Free access (checked October 30, 2026) |
|---|---|---|
| [Exa](https://exa.ai/pricing) | `ExaSearch`, `EXA_API_KEY` | $10 signup bonus plus $10/month; no card required |
| [Tavily](https://www.tavily.com/pricing) | `TavilySearch`, `TAVILY_API_KEY` | 1,000 credits/month ([4,000 for students](https://help.tavily.com/articles/6606514713-student-account)); no card required |
| [Perplexity Search](https://docs.perplexity.ai/docs/getting-started/pricing) | `PerplexitySearch`, `PERPLEXITY_API_KEY` | Pay as you go |
| [OpenAI web search](https://developers.openai.com/api/docs/guides/tools-web-search) | `OpenAISearch`, `OPENAI_API_KEY`, `AGENTKV_SEARCH_MODEL` | Pay as you go |

Each run saves the report, the archived source text, the trace, and token and search receipts under `runs/topic-*/`.

In [ ]:
from datetime import datetime
from research_app import ResearchAgent
from research_app.llm import BedrockLLM
from research_app.search import make_search

RUN_WEB_RESEARCH = False
SEARCH_PROVIDER = "exa"  # or "tavily", "perplexity", "openai"
TOPIC = "What are the tradeoffs between passkeys and passwords for account recovery?"

if RUN_WEB_RESEARCH:
    web_agent = ResearchAgent(llm=BedrockLLM(), search=make_search(SEARCH_PROVIDER), cap=2)
    topic_run = Path("runs") / f"topic-{datetime.now():%Y%m%d-%H%M%S}"
    web_result = await web_agent.research(TOPIC, out=topic_run)
    print(f"Citation checks passed: {web_result.valid}; saved to {topic_run}")
    display(Markdown(web_result.markdown()))
else:
    print("Optional web research is disabled. Configure your providers, then set RUN_WEB_RESEARCH = True.")